# Extended: Enrich from External APIs

Picks up the graph saved by `demo.ipynb` and enriches it with data from:
- **Wikidata**: batch SPARQL query for MusicBrainz artist IDs
- **Discogs**: per-record REST API for first pressing metadata and marketplace prices

Two contrasting enrichment patterns: one federated query vs. many small API calls.

---
## Load the graph

In [ ]:
from maplib import Model
import polars as pl

m = Model()
ns = "http://example.org/music/"

# Read the graph built by demo.ipynb
m.read("data/rdf/music_graph.ttl")
print(f"Loaded graph: {m.size()} triples")

---
## Enrich from Wikidata

Wikidata speaks RDF natively — we can ask its SPARQL endpoint for triples (CONSTRUCT) instead of rows (SELECT), read the Turtle response straight into our graph, and link it to our bands with a single SPARQL INSERT. No JSON parsing, no DataFrames — just RDF in, RDF out.

In [ ]:
import urllib.request, urllib.parse, json

# Get band names to query Wikidata for
names = m.query("""
    PREFIX mu: <http://example.org/music/>
    SELECT ?name WHERE { ?b a mu:Band ; mu:name ?name }
""")["name"].to_list()

# CONSTRUCT query returns RDF triples
values = " ".join(f'"{n}"@en' for n in names)
sparql = f"""
CONSTRUCT {{
    ?item rdfs:label ?label .
    ?item wdt:P434  ?mbid .
}}
WHERE {{
    ?item rdfs:label ?label ;
          wdt:P434  ?mbid .
    VALUES ?label {{ {values} }}
}}
"""

req = urllib.request.Request(
    "https://query.wikidata.org/sparql?query=" + urllib.parse.quote(sparql),
    headers={"Accept": "text/turtle", "User-Agent": "maplib-pdc-demo/1.0"},
)
turtle = urllib.request.urlopen(req).read().decode()

# Read Wikidata triples into the graph (transient = queryable but not serialised)
m.reads(turtle, format="turtle", transient=True)

# Link to our bands: match on name, copy the MusicBrainz ID
m.insert("""
    PREFIX mu:  <http://example.org/music/>
    PREFIX wdt: <http://www.wikidata.org/prop/direct/>
    CONSTRUCT { ?b mu:musicbrainzId ?mbid }
    WHERE {
        ?b a mu:Band ; mu:name ?name .
        ?wd rdfs:label ?label ;
            wdt:P434   ?mbid .
        FILTER(STR(?label) = ?name)
    }
""")

# Verify
m.query("""
    PREFIX mu: <http://example.org/music/>
    SELECT ?name ?mbid
    WHERE { ?b a mu:Band ; mu:name ?name ; mu:musicbrainzId ?mbid }
    ORDER BY ?name
""")

---
## Enrich from Discogs

Discogs has detailed metadata on physical releases: original pressing year, country, label, format, and live marketplace prices. We'll hit their REST API to find the first pressing of each album in our graph.

A different enrichment pattern than Wikidata: instead of one batch SPARQL query, Discogs is a per-record REST API — one request per album.

In [ ]:
DISCOGS_TOKEN = ""  # Get one at discogs.com/settings/developers

base_url = "https://api.discogs.com"
discogs_headers = {
    "Authorization": f"Discogs token={DISCOGS_TOKEN}",
    "User-Agent": "maplib-pdc-demo/1.0",
}

# Get albums from the graph (strip <> from IRIs for later use with map_triples)
albums = m.query("""
    PREFIX mu: <http://example.org/music/>
    SELECT ?album_iri ?title ?artist_name 
    WHERE {
        ?album_iri a mu:Album ;
                   mu:title ?title ;
                   mu:artist ?band .
        ?band mu:name ?artist_name .
    }
""").with_columns(pl.col("album_iri").str.strip_chars("<>"))

import time

pressings = []
for i, row in enumerate(albums.iter_rows(named=True)):
    q = urllib.parse.quote(f"{row['artist_name']} {row['title']}")
    search_url = f"{base_url}/database/search?q={q}&type=release&sort=year&sort_order=asc&per_page=1"
    req = urllib.request.Request(search_url, headers=discogs_headers)
    try:
        resp = json.loads(urllib.request.urlopen(req).read())
    except Exception as e:
        if i == 0:
            print(f"API error: {e} — check your DISCOGS_TOKEN")
            break
        continue

    if not resp.get("results"):
        continue

    release_id = resp["results"][0]["id"]

    # Get full release details (first pressing metadata + pricing)
    rel_url = f"{base_url}/releases/{release_id}"
    req2 = urllib.request.Request(rel_url, headers=discogs_headers)
    try:
        rel = json.loads(urllib.request.urlopen(req2).read())
    except Exception:
        continue

    pressings.append({
        "album_iri": row["album_iri"],
        "title": row["title"],
        "artist": row["artist_name"],
        "discogs_id": str(release_id),
        "press_year": rel.get("year"),
        "press_country": rel.get("country"),
        "press_label": rel["labels"][0]["name"] if rel.get("labels") else None,
        "press_format": rel["formats"][0]["name"] if rel.get("formats") else None,
        "lowest_price": rel.get("lowest_price"),
        "num_for_sale": rel.get("num_for_sale"),
        "have": rel.get("community", {}).get("have"),
        "want": rel.get("community", {}).get("want"),
    })
    print(f"  [{i+1}/{len(albums)}] {row['artist_name']} – {row['title']}")
    time.sleep(1)  # respect Discogs rate limit

if pressings:
    discogs_df = pl.DataFrame(pressings)
    print(f"\nFound first pressings for {len(discogs_df)} / {len(albums)} albums")
    discogs_df.select("artist", "title", "press_year", "press_country", "press_label", "lowest_price")
else:
    discogs_df = None
    print("\nNo results. Set DISCOGS_TOKEN above (get one at discogs.com/settings/developers)")

In [ ]:
# Map first pressing metadata into the graph
if discogs_df is not None:
    m.map_triples(
        discogs_df.select(pl.col("album_iri").alias("subject"), pl.col("discogs_id").alias("object")),
        predicate=ns + "discogsId",
    )

    for col, pred in [
        ("press_year", "firstPressYear"),
        ("press_country", "firstPressCountry"),
        ("press_label", "firstPressLabel"),
        ("press_format", "firstPressFormat"),
        ("lowest_price", "discogsLowestPrice"),
        ("have", "discogsCommunityHave"),
        ("want", "discogsCommunityWant"),
    ]:
        subset = discogs_df.filter(pl.col(col).is_not_null())
        if len(subset) > 0:
            m.map_triples(
                subset.select(pl.col("album_iri").alias("subject"), pl.col(col).alias("object")),
                predicate=ns + pred,
            )

    print(f"Graph now has {m.size()} triples")
else:
    print("Skipped — no Discogs data to map")

---
## Compare prices

Our vinyl store prices vs. Discogs first pressing marketplace prices, all queryable in one SPARQL statement.

In [ ]:
# Compare our vinyl store prices with Discogs first pressing prices
m.query("""
    PREFIX mu: <http://example.org/music/>
    SELECT ?artist ?title ?store_price ?discogs_lowest ?press_year ?press_country ?label
    WHERE {
        ?a a mu:Album ;
           mu:title  ?title ;
           mu:artist ?band ;
           mu:price  ?store_price ;
           mu:discogsLowestPrice ?discogs_lowest .
        ?band mu:name ?artist .
        OPTIONAL { ?a mu:firstPressYear ?press_year }
        OPTIONAL { ?a mu:firstPressCountry ?press_country }
        OPTIONAL { ?a mu:firstPressLabel ?label }
    }
    ORDER BY DESC(?discogs_lowest)
""")

---
## DataFrame back out

In [ ]:
catalog = m.query("""
    PREFIX mu: <http://example.org/music/>
    PREFIX skos: <http://www.w3.org/2004/02/skos/core#>

    SELECT ?name ?genre ?country ?formed ?era ?mbid
    WHERE {
        ?b a mu:Band ;
           mu:name    ?name ;
           mu:genre   ?g ;
           mu:country ?country ;
           mu:formed  ?formed .
        ?g skos:prefLabel ?genre .
        OPTIONAL { ?b mu:era ?era }
        OPTIONAL { ?b mu:musicbrainzId ?mbid }
    }
    ORDER BY ?formed
""")

catalog

In [ ]:
catalog.write_parquet("data/band_catalog_enriched.parquet")
print("Written to data/band_catalog_enriched.parquet")

---
## Explore

The enriched graph now has Wikidata IDs, Discogs metadata, and marketplace prices. Try searching for an album to see all the linked data.

In [ ]:
m.insert("""
    PREFIX mu:   <http://example.org/music/>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    CONSTRUCT { ?b rdfs:label ?name }
    WHERE     { ?b mu:name ?name }
""")

m.insert("""
    PREFIX mu:   <http://example.org/music/>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    CONSTRUCT { ?a rdfs:label ?title }
    WHERE     { ?a mu:title ?title }
""")

server = m.explore(port=7654)

In [ ]:
server.stop()

---
## Recap

What we did:
1. Loaded the graph built by `demo.ipynb`
2. Enriched from Wikidata (batch SPARQL) with MusicBrainz artist IDs
3. Enriched from Discogs (per-record REST API) with first pressing metadata and marketplace prices
4. Compared our store prices with Discogs marketplace
5. Exported an enriched DataFrame to Parquet

Two enrichment patterns:
- **Wikidata**: one SPARQL query, batch result, fast
- **Discogs**: per-record REST calls, rate-limited, richer metadata

Both feed back into the same graph via `map_triples()` and become queryable together.

```
pip install maplib
```

- [maplib docs](https://datatreehouse.github.io/maplib/)
- [maplib on GitHub](https://github.com/DataTreehouse/maplib)
- [Why should you care about Knowledge Graphs?](https://veronahe.substack.com/p/data-engineer-why-should-you-care)